# Databento Options Pull — Wide Strike Band

Pulls OPRA `cbbo-1m` for the wide-strike option universe used in the calendar-straddle earnings backtest, and reduces it to one EOD bid/ask/mid row per `(symbol, date)`.

**You need:**
- A Databento account + API key (https://databento.com)
- The `symbol_list_wide.csv` file shared alongside this notebook

**This costs real money.** A free cost estimate is shown in step 2 before any pull happens. Do not skip it.

In [ ]:
!pip install -q databento pandas pyarrow

In [ ]:
import getpass, os
os.environ["DATABENTO_API_KEY"] = getpass.getpass("Databento API key: ")

## 1. Upload `symbol_list_wide.csv`

Click **Choose Files** and select the CSV that was shared with you.

In [ ]:
from google.colab import files
import pandas as pd

uploaded = files.upload()
symbols = pd.read_csv("symbol_list_wide.csv")["symbol"].tolist()
print(f"Loaded {len(symbols):,} option symbols")

## 2. Cost estimate (FREE — no charge)

`metadata.get_cost` does not spend credits. Use this to confirm the bill before pulling.

In [ ]:
import databento as db

DATASET        = "OPRA.PILLAR"
SCHEMA         = "cbbo-1m"
BACKTEST_START = "2016-01-01"
BACKTEST_END   = "2026-05-01"
BATCH          = 500

client = db.Historical(key=os.environ["DATABENTO_API_KEY"])
total = 0.0
n_batches = (len(symbols) + BATCH - 1) // BATCH
print(f"Querying cost in {n_batches} batches of {BATCH}...")
for i in range(0, len(symbols), BATCH):
    batch = symbols[i:i + BATCH]
    bn = i // BATCH + 1
    try:
        cost = client.metadata.get_cost(
            dataset=DATASET, symbols=batch, stype_in="raw_symbol",
            schema=SCHEMA, start=BACKTEST_START, end=BACKTEST_END,
        )
        total += cost
        print(f"  batch {bn:>3}/{n_batches}: {len(batch)} symbols -> ${cost:,.4f}")
    except Exception as exc:
        print(f"  batch {bn}: ERROR {exc}")

print()
print("=" * 55)
print(f"  ESTIMATED COST : ${total:,.2f} USD")
print("=" * 55)

## 3. Pull the data (CHARGES YOUR DATABENTO ACCOUNT)

Only run the next cell if the estimated cost above is acceptable.

In [ ]:
EOD_UTC_HOUR = 20
PULL_BATCH   = 1000

def pull_in_batches(client, symbols, start, end, batch_size=PULL_BATCH):
    frames = []
    n_batches = (len(symbols) + batch_size - 1) // batch_size
    for i, start_idx in enumerate(range(0, len(symbols), batch_size), 1):
        batch = symbols[start_idx:start_idx + batch_size]
        print(f"  [{i:>3}/{n_batches}] {len(batch)} symbols ...", end=" ", flush=True)
        try:
            store = client.timeseries.get_range(
                dataset=DATASET, symbols=batch, stype_in="raw_symbol",
                schema=SCHEMA, start=start, end=end,
            )
            df = store.to_df(pretty_ts=False, map_symbols=True)
            if df.empty:
                print("0 records")
                continue
            print(f"{len(df):,} raw records")
            frames.append(df)
        except Exception as exc:
            print(f"ERROR: {exc}")

    if not frames:
        return pd.DataFrame()

    combined = pd.concat(frames).reset_index()
    combined["ts_recv"] = pd.to_datetime(combined["ts_recv"], utc=True)
    combined["date"]    = combined["ts_recv"].dt.date
    combined["hour"]    = combined["ts_recv"].dt.hour

    eod = combined[combined["hour"] >= EOD_UTC_HOUR].copy()
    if eod.empty:
        eod = combined.copy()

    eod = (
        eod.sort_values("ts_recv")
        .groupby(["symbol", "date"])
        .last()
        .reset_index()
    )
    eod["bid"] = eod["bid_px_00"].astype(float)
    eod["ask"] = eod["ask_px_00"].astype(float)
    eod["mid"] = (eod["bid"] + eod["ask"]) / 2
    eod = eod[(eod["bid"] > 0) & (eod["ask"] > 0)].copy()
    return eod[["symbol", "date", "bid", "ask", "mid"]].sort_values(
        ["symbol", "date"]
    ).reset_index(drop=True)

print("Pulling data...\n")
opts = pull_in_batches(client, symbols, BACKTEST_START, BACKTEST_END)

if opts.empty:
    print("\nNo data returned.")
else:
    opts.to_parquet("options_daily_wide.parquet", index=False)
    size_mb = os.path.getsize("options_daily_wide.parquet") / 1e6
    print(f"\nSaved {len(opts):,} EOD rows -> options_daily_wide.parquet ({size_mb:.1f} MB)")
    print(f"Unique symbols with data : {opts['symbol'].nunique():,}")
    print(f"Date range               : {opts['date'].min()} -> {opts['date'].max()}")

## 4. Download the result

The parquet lives in Colab's ephemeral filesystem. Run this to pull it to your machine before the runtime is recycled.

In [ ]:
files.download("options_daily_wide.parquet")